# Train, Validation, and Test Thinking

**Project question:** How do I compare models without spending the final test set on model choice?

By the end of this notebook, you should be able to:

- assign distinct training, validation, and test roles
- recognize underfitting and overfitting from error patterns
- evaluate the selected model once on untouched test observations

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [ ]:

from lite_setup import ensure_packages
await ensure_packages()

In [ ]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
df = pd.read_csv(DATA / 'evaluation_regression.csv')
X = df.drop(columns=['id', 'y'])
y = df['y']
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.20, random_state=4031
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_development, y_development, test_size=0.25, random_state=4031
)
{'train': len(X_train), 'validation': len(X_valid), 'test': len(X_test)}

The training set estimates coefficients. The validation set chooses among candidate specifications. The test set remains untouched until one specification has been selected.

In [ ]:
models = {
    'underfit_linear': LinearRegression(),
    'linear_all_predictors': LinearRegression(),
    'quadratic_degree_2': make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression()),
    'high_variance_degree_5': make_pipeline(PolynomialFeatures(degree=5, include_bias=False), LinearRegression()),
}
feature_sets = {
    'underfit_linear': ['x1', 'x2', 'x3'],
    'linear_all_predictors': X.columns.tolist(),
    'quadratic_degree_2': ['x1', 'x2', 'x3'],
    'high_variance_degree_5': ['x1', 'x2', 'x3'],
}

def rmse(actual, predicted):
    return float(np.sqrt(mean_squared_error(actual, predicted)))

rows = []
for name, model in models.items():
    cols = feature_sets[name]
    model.fit(X_train[cols], y_train)
    for split, X_part, y_part in [
        ('train', X_train[cols], y_train),
        ('validation', X_valid[cols], y_valid),
    ]:
        pred = model.predict(X_part)
        rows.append({
            'model': name, 'split': split,
            'rmse': rmse(y_part, pred),
            'mae': mean_absolute_error(y_part, pred),
        })
comparison = pd.DataFrame(rows)
comparison.pivot(index='model', columns='split', values='rmse').sort_values('validation')

**Interpretation:** Training error rewards flexibility. Validation error estimates how the complete fitted workflow transfers to new observations. A large train-validation gap is evidence of variance/overfitting, not proof that the flexible model is always bad.

In [ ]:
validation_rows = comparison.query("split == 'validation'")
selected_name = validation_rows.loc[validation_rows['rmse'].idxmin(), 'model']
selected_columns = feature_sets[selected_name]
final_model = clone(models[selected_name])
final_model.fit(X_development[selected_columns], y_development)
test_pred = final_model.predict(X_test[selected_columns])
final_result = pd.DataFrame([{
    'selected_model': selected_name,
    'test_rmse': rmse(y_test, test_pred),
    'test_mae': mean_absolute_error(y_test, test_pred),
}])
final_result

The final test result is reported once, after selection. If it changes the model choice, the test set has become another validation set and a new external test set would be needed.